# Titanic (Kaggle)
## 1. imports

In [103]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

## 2. Load Data

In [104]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print(train.shape, test.shape)
train.head()

(891, 12) (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 3. Quick checks

In [105]:
train.isna().sum().sort_values(ascending=False).head(10)

Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
Pclass           0
Survived         0
Sex              0
Parch            0
SibSp            0
dtype: int64

## 4. Cleaning + encoding

In [106]:
median_age = train["Age"].median()
train["Age"] = train["Age"].fillna(median_age)
test["Age"] = test["Age"].fillna(median_age)
test["Fare"] = test["Fare"].fillna(train["Fare"].median())
train.drop("Cabin", axis=1, inplace=True)
test.drop("Cabin", axis=1, inplace=True)
train["Embarked"] = train["Embarked"].fillna(train["Embarked"].mode()[0])

train.drop(["PassengerId", "Name", "Ticket"], axis=1, inplace=True)
test.drop(["Name", "Ticket"], axis=1, inplace=True)
train["Sex"] =  train["Sex"].map({"male": 0, "female": 1})
test["Sex"] =  test["Sex"].map({"male": 0, "female": 1})
train["Embarked"] = train["Embarked"].map({"S": 0, "C": 1, "Q": 2})
test["Embarked"] = test["Embarked"].map({"S": 0, "C": 1, "Q": 2})

y = train["Survived"]
X = train.drop("Survived", axis=1)

print("Train features:", X.shape)
print("Test features:", test.shape)

Train features: (891, 7)
Test features: (418, 8)


## 5. Train + local validation

In [107]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

val_acc = model.score(X_val, y_val)
print("Validation accuracy:", val_acc)

Validation accuracy: 0.8044692737430168


## 6. Train on full data + create submission

In [108]:
model.fit(X, y)

test_passenger_id = test["PassengerId"].copy()
X_test = test.drop("PassengerId", axis=1)

test_pred = model.predict(X_test)


submission = pd.DataFrame({
    "PassengerId": test_passenger_id,
    "Survived": test_pred.astype(int)
})

submission_path = Path("../output/submissions/submission.csv")
submission.to_csv(submission_path, index=False)

print("Submission saved to:", submission_path)
submission.head()

Submission saved to: ../output/submissions/submission.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
